In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class MF(object) :
    #   Initializing utility matrix, item features matrix X and user features matrix W, 
    # items bias vector b and users matrix d
    #   lamb - lambda is regularization parameter
    def __init__(self, Y, K, lamb = 0.1,  X_init = None, W_init = None, learning_rate = 0.5, max_iter = 300, print_every = 50) :
        self.Y = Y
        self.K = K
        self.lamb = lamb
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.print_every = print_every
        # Number of users, items and ratings 
        self.n_users = int(np.max(Y[:, 0])) + 1
        self.n_items = int(np.max(Y[:, 1])) + 1
        self.n_ratings = Y.shape[0]
        # Randomly initilize matrixs W and X, bias vectors b and d
        self.X = np.random.rand(K, self.n_items) if X_init is None else X_init
        self.W = np.random.rand(K, self.n_users) if W_init is None else W_init
        self.b = np.random.rand(self.n_items)
        self.d = np.random.rand(self.n_users)
    # General loss function 
    def loss(self) :
        L = 0
        # Get item and user ids of rated items
        for i in range(self.n_ratings) :
            n, m, rating = int(self.Y[i, 0]), int(self.Y[i, 1]), self.Y[i, 2]
            L += (self.X[:, m].T.dot(self.W[:, n]) + self.b[m] + self.d[n] - rating) ** 2
        L = L * 0.5 / self.n_ratings
        # Regularization loss
        regul_loss = 0.5 * self.lamb * (np.sum(self.X ** 2) + np.sum(self.W ** 2))
        return L + regul_loss
    # Fix X and b, optimize W and d
    def update_Wd(self) :
        for n in range(self.n_users) :
            # Get user id from index and find all ratings that this user made
            ids = np.where(self.Y[:, 0] == n)[0]
            items_id, ratings = self.Y[ids, 1].astype(np.int32), self.Y[ids, 2]
            xn = self.X[:, items_id]
            bn = self.b[items_id]
            # Calculating gradient of wn and dn
            for i in range(30) :
                wn = self.W[:, n]
                dn = self.d[n]
                error = xn.T.dot(wn) + bn + dn - ratings
                grad_wn = xn.dot(error) / self.n_ratings + self.lamb * wn
                grad_dn = np.sum(error) / self.n_ratings
                # Update parameters
                self.W[:, n] = wn - self.learning_rate * grad_wn
                self.d[n] = dn - self.learning_rate * grad_dn
    # Fix W and d, optimize X and b
    def update_Xb(self) :
        for m in range(self.n_items) :
            # Get item id from index and find all ratings that rated this item
            ids = np.where(self.Y[:, 1] == m)[0]
            users_id, ratings = self.Y[ids, 0], self.Y[ids, 2]
            wm = self.W[:, users_id]
            dm = self.d[users_id]
            # Calculating gradient of xm and bm
            for i in range(30) :
                xm = self.X[:, m]
                bm = self.b[m]
                error = wm.T.dot(xm) + dm + bm - ratings
                grad_xm = wm.dot(error) / self.n_ratings + self.lamb * xm
                grad_bm = np.sum(error) / self.n_ratings
                # Update parameters
                self.X[:, m] = xm - self.learning_rate * grad_xm
                self.b[m] = bm - self.learning_rate * grad_bm
    # Implementing gradient descent
    def fit(self) :
        for _ in range(self.max_iter) :
            self.update_Wd()
            self.update_Xb()
            if (_ + 1) % self.print_every == 0 :
                loss = self.loss()
                rmse_train = self.evaluate_RMSE(self.Y)
                print('iter = %d, loss = %.4f, RMSE train = %.4f'%(_ + 1, loss, rmse_train))
    # Making predictions
    def predict(self, user, item) :
        n = int(user)
        m = int(item)
        pred = self.X[:, m].T.dot(self.W[:, n]) + self.b[m] + self.d[n]
        return max(0, min(10, pred))
    def evaluate_RMSE(self, rate_test) :
        n_tests = rate_test.shape[0]
        SE = 0
        for n in range(n_tests) :
            pred = self.predict(rate_test[n, 0], rate_test[n, 1])
            SE += (pred - rate_test[n, 2]) ** 2
        SE = SE / n_tests
        RMSE = np.sqrt(SE)
        return RMSE
    def recommend(self, user_id, k_items) :
        pass

        
    

In [ ]:
data = pd.read_csv(r'data\Ratings.csv')
data.head(15)
data = data.iloc[: 20000]


In [5]:
users = pd.unique(data['User-ID'])
items = pd.unique(data['ISBN'])
data['User-ID'] = data['User-ID'].apply(lambda x : np.where(users == x)[0][0])
data['ISBN'] = data['ISBN'].apply(lambda x : np.where(items == x)[0][0])

In [6]:
data = np.array(data)

In [7]:
rs = MF(data, K = 5, max_iter = 80, print_every = 10, learning_rate = 2.0)
rs.fit()

iter = 10, loss = 5.9225, RMSE train = 3.4411
iter = 20, loss = 5.1779, RMSE train = 3.2175
iter = 30, loss = 4.6855, RMSE train = 3.0606
iter = 40, loss = 4.3048, RMSE train = 2.9336
iter = 50, loss = 3.9897, RMSE train = 2.8240
iter = 60, loss = 3.7192, RMSE train = 2.7265
iter = 70, loss = 3.4820, RMSE train = 2.6379
iter = 80, loss = 3.2711, RMSE train = 2.5566
